# C02 · Integer promotion, overflow and saturation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C02_integer_promotion_overflow_and_saturation.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 50 min

**Skill Hub concepts:** Integer promotion and usual arithmetic conversions; Signed overflow; Undefined, unspecified and implementation-defined behavior; Saturating arithmetic

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C02")

## Map: what type does the arithmetic happen in? (32-bit `int`, as on Cortex-M and x86-64)

| Operands | Step 1: integer promotion | Step 2: usual arithmetic conversion | Result type |
|---|---|---|---|
| `uint8_t + uint8_t` | both become `int` | none | `int` (so 200 + 100 = 300, no wrap) |
| `uint16_t * uint16_t` | both become **signed** `int` | none | `int` (65535 * 65535 overflows: undefined) |
| `int + unsigned` | none | `int` converts to `unsigned` | `unsigned` (so `-1 < 1u` is false) |
| `int32_t + int64_t` | none | `int32_t` converts to `int64_t` | `int64_t` |
| `~(uint8_t)x` | becomes `int` first | none | `int` (upper 24 bits become 1) |

| Category | Meaning | Examples |
|---|---|---|
| Undefined | anything may happen; the optimizer assumes it never does | signed overflow, shift by >= width, null dereference, data race |
| Unspecified | one of several allowed results, need not be documented | argument evaluation order, padding byte values |
| Implementation-defined | compiler picks and documents | `sizeof(int)`, signedness of `char`, right shift of a negative number |

Saturating add for $n$-bit signed values: $\mathrm{sat}(a+b) = \min\!\big(\max(a+b,\,-2^{n-1}),\; 2^{n-1}-1\big)$, computed
without ever overflowing.

## Task 1: predict, then let the compiler grade you
Each key is a C expression evaluated with `uint8_t a = 200, b = 100; uint32_t u = 1; unsigned short s = 0xFFFF;`
and printed with `%d`. Fill in the integers you expect.

In [ ]:
predictions = {
        "a + b": None,
        "(uint8_t)(a + b)": None,
        "-1 < 1u": None,
        "(int)sizeof(a + b)": None,
        "u - 2 > 0": None,
        "(int)sizeof('a')": None,
        "~s": None,
}
lab.check("predict", predictions)

## Task 2: saturating arithmetic that never overflows
The harness runs with the undefined-behavior sanitizer, so `a + b` on two `int32_t` near the limits fails.

In [ ]:
%%writefile sat.c
#include <stdint.h>

int16_t sat_add_i16(int16_t a, int16_t b) {
    return (int16_t)(a + b); /* TODO */
}

uint8_t sat_add_u8(uint8_t a, uint8_t b) {
    return (uint8_t)(a + b); /* TODO */
}

int32_t sat_add_i32(int32_t a, int32_t b) {
    return a + b; /* TODO: check against the limits BEFORE adding */
}

In [ ]:
lab.check("saturate")

## Task 3: name the category
Use `"undefined"`, `"unspecified"` or `"implementation-defined"`.

In [ ]:
categories = {
        "signed int overflow": None,
        "order in which f(g(), h()) evaluates g and h": None,
        "right shift of a negative int": None,
        "x << 32 where x is uint32_t": None,
        "whether plain char is signed": None,
        "modifying a string literal": None,
        "values of padding bytes in a struct": None,
        "converting 300 to int8_t": None,
}
lab.check("categories", categories)

The optimizer really uses "signed overflow never happens". Compare the two functions at `-O2`:

In [ ]:
from roboprep import clab
open("ub.c", "w").write("int will_wrap_s(int x) { return x + 1 < x; }\n"
                        "int will_wrap_u(unsigned x) { return x + 1u < x; }\n")
asm = clab.sh("gcc -O2 -S -fno-asynchronous-unwind-tables -fcf-protection=none -o - ub.c").stdout
print("\n".join(l for l in asm.splitlines() if not l.startswith("\t.")))

## Checkpoint: Math
**M1 (Apply).** `uint8_t r = 200 + 100;` value of `r`?

**M2 (Apply).** A 16-bit tick counter: `(uint16_t)(5u - 65530u)` ticks elapsed?

**M3 (Apply).** `sat_add_i16(30000, 10000)`?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** `uint16_t a = 0xFFFF; uint32_t r = a * a;` on a 32-bit `int` target. What is wrong?
- a) nothing: unsigned arithmetic wraps
- b) both operands promote to signed `int`, and 4294836225 overflows `int`: undefined behavior
- c) the result is truncated to 16 bits before assignment

**K2 (Evaluate).** Explain the three behavior categories and why undefined behavior can delete code you wrote
(use `x + 1 < x` from above).

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c02.h`).
3. In `roboprep/checks/c02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()